# Module 9(b) — Magnitude pruning of the dense fan model

We take the Module 7 dense classifier (13 features → 8 → 16 → n_classes) and prune it: zero out the smallest-magnitude weights during a fine-tuning run, sweep the target **sparsity**, and plot sparsity vs accuracy.

Along the way we answer the two questions that matter on an MCU:
1. How many weights can go before accuracy notices?
2. What does that actually buy — and what does it *not* buy (spoiler: unstructured zeros don't speed up a dense kernel)?

> **VERIFY — tfmot API currency:** `tensorflow-model-optimization` (tfmot) requires the legacy `tf.keras` (Keras 2). With TensorFlow ≥ 2.16 (Keras 3 default) install `tf_keras` and set `TF_USE_LEGACY_KERAS=1` **before importing tensorflow**, or pin `tensorflow<2.16`. tfmot has had minimal maintenance since ~2023 — re-run this notebook against the course Python environment before teaching.

In [ ]:
import os
os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')   # harmless on TF<2.16; required on >=2.16 with tf_keras installed

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import tensorflow_model_optimization as tfmot

print('TF', tf.__version__, '| tfmot', tfmot.__version__)

## 1. Data — the Module 7 fan features

Same loader, same 13 features, same scaling story as Module 7 (NNs need the StandardScaler; remember to export offset/scale if you deploy).

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('../../module7-models/rf-features'))
from train_rf import load_windows   # noqa: E402

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

DATA_DIR = '../../module7-models/rf-features/data/raw'
X_raw, X_feat, y, groups, _ = load_windows(DATA_DIR)

le = LabelEncoder()
y_int = le.fit_transform(y)
n_classes = len(le.classes_)
print('classes:', list(le.classes_))

Xtr, Xte, ytr, yte = train_test_split(X_feat, y_int, test_size=0.25,
                                      random_state=42, stratify=y_int)
scaler = StandardScaler().fit(Xtr)
Xtr, Xte = scaler.transform(Xtr), scaler.transform(Xte)
ytr_oh = keras.utils.to_categorical(ytr, n_classes)
yte_oh = keras.utils.to_categorical(yte, n_classes)
print('train/test:', Xtr.shape, Xte.shape)

## 2. Baseline — the unpruned dense model

In [ ]:
def build_model():
    return keras.Sequential([
        keras.layers.Input(shape=(Xtr.shape[1],)),
        keras.layers.Dense(8, activation='relu'),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(n_classes, activation='softmax'),
    ])

baseline = build_model()
baseline.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                   restore_best_weights=True)
baseline.fit(Xtr, ytr_oh, epochs=150, batch_size=32, validation_split=0.2,
             callbacks=[es], verbose=0)
base_acc = baseline.evaluate(Xte, yte_oh, verbose=0)[1]
print(f'baseline test accuracy: {base_acc:.3f}')
baseline.summary()

In [ ]:
# Motivation picture: most trained weights hug zero.
w_all = np.concatenate([w.flatten() for w in baseline.get_weights() if w.ndim == 2])
plt.figure(figsize=(7, 3))
plt.hist(w_all, bins=60)
plt.title(f'Weight distribution ({len(w_all)} weights) - the freeloaders around 0')
plt.xlabel('weight value'); plt.show()

## 3. Magnitude pruning with a sparsity sweep

`prune_low_magnitude` wraps each layer; during fine-tuning a `PolynomialDecay` schedule gradually raises sparsity from 0 to the target, zeroing the smallest |w| and letting the survivors heal. `strip_pruning` afterwards removes the training wrappers, leaving an ordinary Keras model whose weight matrices simply contain zeros.

In [ ]:
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

EPOCHS = 60
BATCH = 32
steps_per_epoch = int(np.ceil(len(Xtr) * 0.8 / BATCH))
end_step = steps_per_epoch * EPOCHS

sparsities = [0.5, 0.7, 0.8, 0.9]
results = {}
pruned_models = {}

for s in sparsities:
    schedule = tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0, final_sparsity=s,
        begin_step=0, end_step=end_step)

    m = build_model()
    m.set_weights(baseline.get_weights())          # start from the trained model
    pm = prune_low_magnitude(m, pruning_schedule=schedule)
    pm.compile(optimizer=keras.optimizers.Adam(5e-4),
               loss='categorical_crossentropy', metrics=['accuracy'])
    pm.fit(Xtr, ytr_oh, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2,
           callbacks=[tfmot.sparsity.keras.UpdatePruningStep()], verbose=0)

    final = tfmot.sparsity.keras.strip_pruning(pm)
    acc = final.evaluate(Xte, yte_oh, verbose=0)[1]
    # actual achieved sparsity
    ws = [w for w in final.get_weights() if w.ndim == 2]
    actual = np.mean([np.mean(w == 0) for w in ws])
    results[s] = (acc, actual)
    pruned_models[s] = final
    print(f'target {s:.0%} -> achieved {actual:.1%} zeros, test acc {acc:.3f}')

In [ ]:
xs = [0.0] + sparsities
ys = [base_acc] + [results[s][0] for s in sparsities]
plt.figure(figsize=(7, 4))
plt.plot(xs, ys, 'o-')
plt.axhline(base_acc, color='gray', ls=':', label=f'baseline {base_acc:.3f}')
plt.xlabel('target sparsity (fraction of weights zeroed)')
plt.ylabel('test accuracy')
plt.title('Sparsity vs accuracy - dense fan model')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4. What did we actually win?

Unstructured zeros do **not** shrink the dense weight array and do **not** skip MACs in a standard TFLM/EON/AIfES kernel. Where they *do* pay immediately is **compression** — sparse tensors gzip beautifully (relevant for OTA updates and flash storage of compressed models).

In [ ]:
import gzip, io, tempfile

def gzipped_size(model):
    # serialise weights only (comparable across models), then gzip
    buf = io.BytesIO()
    np.savez(buf, *model.get_weights())
    return len(gzip.compress(buf.getvalue()))

print(f'{"model":>12s} {"gzipped weights":>16s} {"test acc":>9s}')
print(f'{"baseline":>12s} {gzipped_size(baseline):>13d} B {base_acc:>9.3f}')
for s in sparsities:
    print(f'{f"{s:.0%} sparse":>12s} {gzipped_size(pruned_models[s]):>13d} B {results[s][0]:>9.3f}')

## 5. Discussion & exercises

1. **Read your curve.** Where is the knee? For a ~300-weight model, is an 80 % cut sensible — or is the honest conclusion "this model was already tiny"?
2. **Structured alternative.** Instead of pruning, retrain with `Dense(4)`/`Dense(8)` layers (half the neurons). Compare accuracy *and* MAC count with the 50 %-unstructured model. Which would actually run faster on the RAK4631, and why?
3. **Prune → quantise.** Take your best pruned model through TFLite int8 PTQ (`tf.lite.TFLiteConverter`, representative dataset = training windows) and compare the gzipped `.tflite` sizes. Do the savings compound?
4. **Where pruning really lives:** big CNNs. The Module 7 CNN's `Flatten(560) → Dense(8)` layer holds ~93 % of its parameters. Prune *that* (or make it structured: fewer filters before Flatten) and report.

**Take-away:** pruning is a size/compression tool by default and a speed tool only when *structured*. For models this small, right-sizing the architecture (Module 7 grid search) usually beats pruning — knowing that is the point of the exercise.